In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import anndata as ad
adata = ad.read_h5ad("./data/dentate_gyrus/dentategyrus_velocity_processed.h5ad")

In [ ]:
adata

In [ ]:
import numpy as np

# Convert to dense arrays first
X = (adata.layers["spliced"] + adata.layers["unspliced"]).toarray()
V = adata.layers["velocity"]

# Column-wise standardization
X = X / X.std(axis=0, keepdims=True)
V = V / V.std(axis=0, keepdims=True)

In [ ]:
from scripts.VectorFieldEmbedder import *
from scripts.plotting import *

emb = VectorFieldEmbedder(X, V, dist_method="phase",
                          dof=30,
                          embed_kwargs={"n_neighbors":50,
                                         "min_dist":0.7})
emb.initialize_embedding(seed=0)
plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    scatter_color=list(adata.obs["clusters"]),
    grid_density=1.0, 
    stream_density=1.2,
    scatter_size=200,
    scatter_alpha=0.2,
    figsize=(8, 8),
    aspect=1,
    vmin=0.0,
    vmax=1.0,
    cmap="tab10",
    grid_size=50
)

In [ ]:
from scripts.pseudotime import *

pseudotime_model = StochasticPseudotime(X=emb.X_emb, vector_field=emb.tps_vf, tps=emb.tps)
root = pseudotime_model.find_root(n_simulations_per_cell=5)
tau = pseudotime_model.compute_pseudotime(roots=root)

In [ ]:
from numpy import log1p  # log(1 + x), avoids issues with zeros
tau_transformed = log1p(tau)
tau_transformed /= np.nanmax(tau_transformed)  # normalize to [0, 1]

plot_velocity_streamplot(
    X_2d=emb.X_emb,
    tps_vf=emb.tps_vf,
    scatter_color=tau_transformed,
    grid_density=1.0, 
    stream_density=1.3,
    scatter_size=200,
    scatter_alpha=0.1,
    figsize=(6, 6),
    aspect=1,
    vmin=0.0,
    vmax=1.0,
    grid_size=50
)

In [ ]:
from scripts.tol_vec_percent import *

# 1) detect
fps = find_fixed_points_grid(emb.tps_vf, emb.X_emb,
                                       grid_size=30)

# 2) build the grid once (so Jacobian fits reuse it)
Xg, Vg = compute_velocity_on_grid(emb.X_emb, emb.tps_vf, grid_size=120)

# 3) loop through candidate fixed points
labels = []
for i, fp in enumerate(fps):
    J = jacobian_from_grid(emb.tps_vf, fp, Xg, Vg, radius=10)
    label = classify_fixed_point(J)
    labels.append(label)
    print(f"FP{i}: {np.round(fp,3)}  →  {label}")

In [ ]:
from matplotlib.lines import Line2D

# Only keep fixed points 0, 1, 2
selected_indices = [0, 1, 2]
fps_subset    = [fps[i] for i in selected_indices]
labels_subset = [labels[i] for i in selected_indices]

Xg, Vg = compute_velocity_on_grid(emb.X_emb, emb.tps_vf, grid_size=30,
                                  margin_ratio=0)

plt.figure(figsize=(5, 5))

# Grey scatter for cells
plt.scatter(emb.X_emb[:, 0], emb.X_emb[:, 1], s=80, color='gray', alpha=0.05)

# Grid quiver
plt.quiver(
    Xg[:, 0], Xg[:, 1],
    Vg[:, 0], Vg[:, 1],
    angles="xy", scale_units="xy", scale=3.0,
    width=0.0025, headwidth=3,
    color="black", alpha=0.9
)

# Fixed points with red dots and white numbers
for i, (fp, label) in zip(selected_indices, zip(fps_subset, labels_subset)):
    plt.text(fp[0], fp[1], f"{i}", color="red", fontsize=12,
             ha='center', va='center', zorder=4)

# Text-only legend for selected fix points
legend_labels = [f"{i}: {labels[i]}" for i in selected_indices]
legend_handles = [Line2D([0], [0], color='none', label=txt) for txt in legend_labels]
plt.legend(handles=legend_handles, loc='upper right', fontsize=9, frameon=False)

plt.axis("equal")
plt.xticks([])
plt.yticks([])
plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

X_emb = emb.X_emb
tps_all = ThinPlateSpline(X_emb)
tps_all.fit(X, dof=30)

# --- Predict expression ---
X_pred = tps_all.predict(X_emb)

# Velocity: (n_cells, d)
v = emb.V_emb
v_norm = np.linalg.norm(v, axis=1, keepdims=True)
v_unit = v / (v_norm + 1e-8)  # normalized, shape (n, d)

# Jacobians: (n_cells, n_genes, d_pca) → project to gene space
J = tps_all.compute_jacobians(emb.X_emb)                 # (cells × genes_valid × d)

# --- Vectorized decomposition ---

# Dot product per cell per gene: shape (n_cells, n_genes)
aligned_component = np.einsum("nd,ngd->ng", v_unit, J)

# Flow-aligned component: shape (n_cells, n_genes, d)
aligned = aligned_component[:, :, None] * v_unit[:, None, :]  # broadcast dot back along velocity

# Orthogonal component: shape (n_cells, n_genes, d)
orthogonal = J - aligned

# Store per-cell aligned and orthogonal components if needed
aligned_components = aligned
orthogonal_components = orthogonal

# Reduce to gene-level scores
aligned_scores = np.mean(aligned_component, axis=0)  # (n_genes,)
orthogonal_scores = np.mean(np.linalg.norm(orthogonal, axis=2), axis=0)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

gene_names = adata.var_names
# ── Thresholds ──
highlight_mask = (np.abs(aligned_scores) > 0.1) | (orthogonal_scores > 0.2)
highlight_x = aligned_scores[highlight_mask]
highlight_y = orthogonal_scores[highlight_mask]
highlight_genes = np.array(gene_names)[highlight_mask]

# ── Plot ──
plt.figure(figsize=(6, 5))
plt.scatter(aligned_scores, orthogonal_scores, s=10, alpha=0.7, edgecolors='none')
plt.scatter(highlight_x, highlight_y, color='crimson', s=30, label='Highlighted genes')

# Draw labels
for x, y, gene in zip(highlight_x, highlight_y, highlight_genes):
    plt.text(x, y, gene, fontsize=9, color='crimson', ha='left', va='bottom')

plt.axhline(0, color='grey', linestyle='--', linewidth=1)
plt.axvline(0, color='grey', linestyle='--', linewidth=1)
plt.xlabel("Flow-aligned score (Regime 1)", fontsize=12)
plt.ylabel("Flow-orthogonal score (Regime 2)", fontsize=12)
plt.title("Gene regime dynamics", fontsize=14)
plt.grid(True, linestyle='--', alpha=0.3)
plt.tight_layout()
plt.show()